In [1]:
!pip install datasets pandas -q

from datasets import load_dataset
import pandas as pd
import random

random.seed(42)


[notice] A new release of pip is available: 24.1 -> 26.1
[notice] To update, run: pip install --upgrade pip


In [30]:
import re
from datasets import load_dataset
import pandas as pd

def extract_boxed_answer(solution):
    if not isinstance(solution, str):
        return None
    
    key = r"\boxed{"
    start = solution.rfind(key)
    
    if start == -1:
        key = r"\fbox{"
        start = solution.rfind(key)
        if start == -1:
            return None
    
    i = start + len(key)
    brace_count = 1
    answer = ""
    
    while i < len(solution):
        char = solution[i]
        
        if char == "{":
            brace_count += 1
            answer += char
        elif char == "}":
            brace_count -= 1
            if brace_count == 0:
                return answer.strip()
            answer += char
        else:
            answer += char
        
        i += 1
    
    return None

def clean_answer(ans):
    if ans is None:
        return None
    
    ans = ans.strip()
    ans = ans.replace(r"\$", "")
    ans = ans.replace("$", "")
    
    if re.fullmatch(r"[0-9,]+", ans):
        ans = ans.replace(",", "")
    
    return ans.strip()
    
math_subjects = ["algebra", "counting_and_probability", "geometry"]

math_rows = []

for subject in math_subjects:
    ds = load_dataset("EleutherAI/hendrycks_math", subject)
    df = ds["train"].to_pandas()
    
    df = df[df["level"] == "Level 5"].copy()
    df["domain"] = "mathematical_reasoning"
    df["source"] = "hendrycks_math"
    df["subject"] = subject
    
    df["final_answer_raw"] = df["solution"].apply(extract_boxed_answer)
    df["final_answer"] = df["final_answer_raw"].apply(clean_answer)
    df = df.rename(columns={
        "problem": "question"
    })
    
    math_rows.append(df[[
        "domain", "source", "subject", "level",
        "question", "solution", "final_answer"
    ]])

math_df = pd.concat(math_rows, ignore_index=True)

sampled_df = (
    math_df
    .groupby("subject", group_keys=False)
    .apply(lambda x: x.sample(n=3, random_state=42))
    .reset_index(drop=True)
)
math_json = []

for i, row in sampled_df.iterrows():
    math_json.append({
        "id": f"MATH_{i+1}",
        "type": "open-ended",
        "question": row["question"],
        "choices": None,
        "answer": row["final_answer"],
        "explanation": row["solution"],
        "difficulty": "Level 5",
        "category": row["subject"],
        "source": "hendrycks_math"
    })
print(math_json) 


[{'id': 'MATH_1', 'type': 'open-ended', 'question': 'Let $f(x) = 3x^2-2$ and $g(f(x)) = x^2 + x +1$.  Find the sum of all possible values of $g(25)$.', 'choices': None, 'answer': '20', 'explanation': "We don't know $g(x)$, so we don't have an expression we can simply stick $25$ in to get an answer. We do, however, know that $g(f(x)) =x^2 + x + 1$.  So, if we can figure out what to put into $f(x)$ such that $25$ is the resulting output, we can use our expression for $g(f(x))$ to find $g(25)$.\n\nIf $f(x) = 25$, then we have $3x^2 - 2 = 25$, so $x^2 = 9$, which means $x=3$ or $x=-3$.  Since $x$ could be $3$ or $-3$, we could have $g(25) = g(f(3))$ or $g(25) = g(f(-3))$.  Using the given expression for $g(f(x))$, the two possible values of $g(25)$ are  $g(f(3)) = 3^2 + 3 + 1 = 13$ and $g(f(-3)) = (-3)^2 + (-3) + 1 = 7$.  The sum of these is $13+7=\\boxed{20}$.", 'difficulty': 'Level 5', 'category': 'algebra', 'source': 'hendrycks_math'}, {'id': 'MATH_2', 'type': 'open-ended', 'question': 

/var/folders/_m/bd0cv3552fzdzs87pkcc1h480000gn/T/ipykernel_18168/3589115899.py:82: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=3, random_state=42))


In [31]:
from datasets import load_dataset
import random
import json

dataset = load_dataset("Idavidrein/gpqa", "gpqa_diamond", split="train")

filtered = [
    x for x in dataset
    if x["High-level domain"] == "Physics"
    and x["Writer's Difficulty Estimate"] is not None
    and ("Post-graduate" in x["Writer's Difficulty Estimate"]
            or "Hard graduate" in x["Writer's Difficulty Estimate"])
]

print("Filtered:", len(filtered))

random.seed(42)
samples = random.sample(filtered, 3)
gpqa_json = []

for i, x in enumerate(samples, start=1):
    
    choices = [
        x["Correct Answer"],
        x["Incorrect Answer 1"],
        x["Incorrect Answer 2"],
        x["Incorrect Answer 3"]
    ]
    
    random.shuffle(choices)
    
    answer_letter = ["A", "B", "C", "D"][choices.index(x["Correct Answer"])]
    
    gpqa_json.append({
        "id": f"GPQA_{i}",
        "type": "multiple-choice",
        "question": x["Question"],
        "choices": {
            "A": choices[0],
            "B": choices[1],
            "C": choices[2],
            "D": choices[3]
        },
        "answer": answer_letter,
        "explanation": x["Explanation"],
        "difficulty": "Hard (Post-graduate level)",
        "category": "Physics",
        "subdomain": x["Subdomain"],
        "source": "GPQA"
    })

with open("gpqa_physics_3.json", "w", encoding="utf-8") as f:
    json.dump(gpqa_json, f, indent=2, ensure_ascii=False)

print("Saved to gpqa_physics_3.json")

print(json.dumps(gpqa_json, indent=2, ensure_ascii=False))

Filtered: 33
Saved to gpqa_physics_3.json
[
  {
    "id": "GPQA_1",
    "type": "multiple-choice",
    "question": "Your colleague has devised a new quantum field theory on four-dimensional spacetime, and is exploring the regularization of certain higher-order loop diagrams in that theory. On their desk you spy a scribbled note: a Feynman diagram, and next to it, the words \"size estimate\" followed by a string of physical constants, numbers, and ratios between what appear to be energy scales. The symbols read: alpha^3 * g^2 sqrt(2) * 8 * 1/(4pi)^6 * (Q/M)^2.\n\nThe diagram, unfortunately, has been rendered unreadable by a spilled coffee. How many loops did it contain?",
    "choices": {
      "A": "6",
      "B": "2",
      "C": "3",
      "D": "1"
    },
    "answer": "C",
    "explanation": "\"The procedure which the colleague was performing is known as power-counting, and is intended to estimate the size of a Feynman diagram before more accurate, but much more difficult, loop integ

In [36]:
from datasets import load_dataset
import itertools
import random
import json

ds = load_dataset(
    "deepmind/code_contests",
    split="train",
    streaming=True
)

N = 5000  
filtered_code = []

for x in itertools.islice(ds, N):
    if (
        x["source"] == 2  # Codeforces
        and x.get("cf_rating") is not None
        and x["cf_rating"] >= 2000
    ):
        filtered_code.append(x)

print("Filtered (>=2000):", len(filtered_code))

random.seed(42)
samples = random.sample(filtered_code, 3)

cf_json= []

for i, x in enumerate(samples, start=1):
    cf_json.append({
        "id": f"CF_{i}",
        "type": "coding",
        "question": x["description"],
        "choices": None,
        "answer": None,
        "solution": x["solutions"]["solution"][0] if len(x["solutions"]["solution"]) > 0 else None,
        "difficulty": f"Codeforces rating {x['cf_rating']}",
        "category": x["cf_tags"],
        "source": "Codeforces",
        "metadata": {
            "name": x["name"],
            "cf_rating": x["cf_rating"],
            "cf_index": x["cf_index"]
        }
    })

with open("codeforces_2000plus_3.json", "w", encoding="utf-8") as f:
    json.dump(cf_json, f, indent=2, ensure_ascii=False)

print("Saved codeforces_2000plus_3.json")



Resolving data files:   0%|          | 0/39 [00:00<?, ?it/s]

Filtered (>=2000): 1325
Saved codeforces_2000plus_3.json


In [37]:
final_dataset = math_json + gpqa_json +     cf_json
with open("final_dataset.json", "w", encoding="utf-8") as f:
    json.dump(final_dataset, f, indent=2, ensure_ascii=False)

print("Saved final_dataset.json")
print("Total samples:", len(final_dataset))
print(json.dumps(final_dataset, indent=2, ensure_ascii=False))


Saved final_dataset.json
Total samples: 15
[
  {
    "id": "MATH_1",
    "type": "open-ended",
    "question": "Let $f(x) = 3x^2-2$ and $g(f(x)) = x^2 + x +1$.  Find the sum of all possible values of $g(25)$.",
    "choices": null,
    "answer": "20",
    "explanation": "We don't know $g(x)$, so we don't have an expression we can simply stick $25$ in to get an answer. We do, however, know that $g(f(x)) =x^2 + x + 1$.  So, if we can figure out what to put into $f(x)$ such that $25$ is the resulting output, we can use our expression for $g(f(x))$ to find $g(25)$.\n\nIf $f(x) = 25$, then we have $3x^2 - 2 = 25$, so $x^2 = 9$, which means $x=3$ or $x=-3$.  Since $x$ could be $3$ or $-3$, we could have $g(25) = g(f(3))$ or $g(25) = g(f(-3))$.  Using the given expression for $g(f(x))$, the two possible values of $g(25)$ are  $g(f(3)) = 3^2 + 3 + 1 = 13$ and $g(f(-3)) = (-3)^2 + (-3) + 1 = 7$.  The sum of these is $13+7=\\boxed{20}$.",
    "difficulty": "Level 5",
    "category": "algebra",
 

In [12]:
import re
import json
import pandas as pd
from datasets import load_dataset

def extract_boxed_answer(solution):
    if not isinstance(solution, str):
        return None

    for key in [r"\boxed{", r"\fbox{"]:
        start = solution.rfind(key)
        if start != -1:
            break
    else:
        return None

    i = start + len(key)
    brace_count = 1
    answer = ""

    while i < len(solution):
        char = solution[i]

        if char == "{":
            brace_count += 1
            answer += char
        elif char == "}":
            brace_count -= 1
            if brace_count == 0:
                return answer.strip()
            answer += char
        else:
            answer += char

        i += 1

    return None

def clean_answer(ans):
    if ans is None:
        return None

    ans = ans.strip()
    ans = ans.replace(r"\$", "").replace("$", "")

    if re.fullmatch(r"[0-9,]+", ans):
        ans = ans.replace(",", "")

    return ans.strip()

math_test_subjects = [
    "algebra",
    "geometry",
    "counting_and_probability",
    "number_theory",
    "intermediate_algebra"
]

math_test_rows = []

for subject in math_test_subjects:
    ds = load_dataset("EleutherAI/hendrycks_math", subject)
    df = ds["train"].to_pandas()

    df = df[df["level"] == "Level 5"].copy()

    df["domain"] = "mathematical_reasoning"
    df["source"] = "hendrycks_math"
    df["subject"] = subject

    df["final_answer_raw"] = df["solution"].apply(extract_boxed_answer)
    df["final_answer"] = df["final_answer_raw"].apply(clean_answer)

    df = df.rename(columns={"problem": "question"})

    math_test_rows.append(df[[
        "domain", "source", "subject", "level",
        "question", "solution", "final_answer"
    ]])

math_test_df = pd.concat(math_test_rows, ignore_index=True)

sampled_math_test_df = (
    math_test_df
    .groupby("subject", group_keys=False)
    .apply(lambda x: x.sample(n=3, random_state=100))
    .reset_index(drop=True)
)

math_test_json = []

for i, row in sampled_math_test_df.iterrows():
    math_test_json.append({
        "id": f"MATH_TEST_{i+1}",
        "type": "open-ended",
        "question": row["question"],
        "choices": None,
        "answer": row["final_answer"],
        "explanation": row["solution"],
        "difficulty": "Level 5",
        "category": row["subject"],
        "source": "hendrycks_math"
    })

with open("math_test_dataset.json", "w", encoding="utf-8") as f:
    json.dump(math_test_json, f, indent=2, ensure_ascii=False)

print("Saved math_test_dataset.json")


number_theory/train-00000-of-00001.parqu(…):   0%|          | 0.00/309k [00:00<?, ?B/s]

number_theory/test-00000-of-00001.parque(…):   0%|          | 0.00/182k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/869 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/540 [00:00<?, ? examples/s]

intermediate_algebra/train-00000-of-0000(…):   0%|          | 0.00/575k [00:00<?, ?B/s]

intermediate_algebra/test-00000-of-00001(…):   0%|          | 0.00/395k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1295 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/903 [00:00<?, ? examples/s]

Saved math_test_dataset.json


/var/folders/_m/bd0cv3552fzdzs87pkcc1h480000gn/T/ipykernel_18168/3521225480.py:86: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=3, random_state=100))


In [28]:
import random
import json

random.seed(100)

domains = ["Physics", "Biology", "Chemistry"]

gpqa_test_json = []
counter=1

for domain in domains:
    
    domain_pool = [
        x for x in dataset
        if x["High-level domain"] == domain
        and isinstance(x.get("Writer's Difficulty Estimate"), str)
        and (
            "Post-graduate" in x["Writer's Difficulty Estimate"]
            or "Hard graduate" in x["Writer's Difficulty Estimate"]
        )
    ]
    
    print(f"{domain} pool size:", len(domain_pool))
    
    samples = random.sample(domain_pool, 3)
    
    for i, x in enumerate(samples, start=1):
        
        choices = [
            x["Correct Answer"],
            x["Incorrect Answer 1"],
            x["Incorrect Answer 2"],
            x["Incorrect Answer 3"]
        ]
        
        random.shuffle(choices)
        answer_letter = ["A", "B", "C", "D"][choices.index(x["Correct Answer"])]
        
        gpqa_test_json.append({
            "id": f"GPQA_TEST_{counter}",
            "type": "multiple-choice",
            "question": x["Question"],
            "choices": {
                "A": choices[0],
                "B": choices[1],
                "C": choices[2],
                "D": choices[3]
            },
            "answer": answer_letter,
            "explanation": x["Explanation"],
            "difficulty": "Hard (Post-graduate level)",
            "category": domain,
            "subdomain": x["Subdomain"],
            "source": "GPQA"
        })
        counter+=1        


with open("gpqa_test_dataset.json", "w", encoding="utf-8") as f:
    json.dump(gpqa_test_json, f, indent=2, ensure_ascii=False)

print("Saved gpqa_test_dataset.json")
print("Total samples:", len(gpqa_test_json))

Physics pool size: 33
Biology pool size: 7
Chemistry pool size: 33
Saved gpqa_test_dataset.json
Total samples: 9


In [38]:
random.seed(100)  

cf_test_samples = random.sample(filtered_code, 3)

cf_test_json = []

for i, x in enumerate(cf_test_samples, start=1):
    cf_test_json.append({
        "id": f"CF_TEST_{i}",
        "type": "coding",
        "question": x["description"],
        "choices": None,
        "answer": None,
        "solution": x["solutions"]["solution"][0] if len(x["solutions"]["solution"]) > 0 else None,
        "difficulty": f"Codeforces rating {x['cf_rating']}",
        "category": x["cf_tags"],
        "source": "Codeforces"
    })
with open("cf_test_json.json", "w", encoding="utf-8") as f:
    json.dump(cf_test_json, f, indent=2, ensure_ascii=False)

print("Saved cf_test_json.json")


Saved cf_test_json.json


In [39]:
test_dataset = math_test_json + gpqa_test_json +     cf_test_json
with open("test_dataset.json", "w", encoding="utf-8") as f:
    json.dump(test_dataset, f, indent=2, ensure_ascii=False)

print("Saved test_dataset.json")
print("Total samples:", len(test_dataset))
print(json.dumps(test_dataset, indent=2, ensure_ascii=False))


Saved test_dataset.json
Total samples: 27
[
  {
    "id": "MATH_TEST_1",
    "type": "open-ended",
    "question": "What is the range of the function $f(x) = \\frac{1}{x^2}$?",
    "choices": null,
    "answer": "(0,\\infty)",
    "explanation": "Note that $f(x) = \\frac{1}{x^2} >0$ for all nonzero $x$. That is, the range of $f$ must only include positive numbers. Conversely, if $a$ is a positive number, then \\[f\\left(\\frac{1}{\\sqrt{a}}\\right)=\\frac{1}{(1/\\sqrt{a})^2} = a,\\]so $a$ is indeed in the range of $f$. Thus, the range of $f$ is the set of all positive real numbers; in interval notation, that's $\\boxed{(0,\\infty)}$.",
    "difficulty": "Level 5",
    "category": "algebra",
    "source": "hendrycks_math"
  },
  {
    "id": "MATH_TEST_2",
    "type": "open-ended",
    "question": "Given that\n\n\\begin{align*}\n\\frac{1}{x}+\\frac{1}{y}&=3,\\\\\nxy+x+y&=4,\n\\end{align*}\n\ncompute $x^2y+xy^2$.",
    "choices": null,
    "answer": "3",
    "explanation": "The first equa